In [ ]:
!pip install -q zstandard requests

In [ ]:
import os

N_POSITIONS   = 30_000_000   # labelled positions to collect (RAM no longer
                             # limits this -- training streams one shard at a
                             # time, so 50M or 100M work too; only preprocessing
                             # time and disk grow)
SHARD_SIZE    = 1_000_000    # positions per saved .npz shard
HIDDEN        = 256          # feature-transformer width. Once the pipeline
                             # works end-to-end, try 512: noticeably stronger,
                             # at the cost of a somewhat slower engine eval.
BATCH_SIZE    = 16_384
EPOCHS        = 25
LR            = 1e-3
LR_DROP_EVERY = 8            # epochs between learning-rate /= 3
CP_CLAMP      = 2000         # clamp targets to +-2000 cp; mates become +-2000
VAL_MAX       = 200_000      # positions from the held-out shard used for val
SEED          = 42

DATA_DIR   = "/content/nnue_data"
CKPT_DIR   = "/content/nnue_ckpt"
OUT_FILE   = "/content/mihir_v1.nnue"
LICHESS_URL = "https://database.lichess.org/lichess_db_eval.jsonl.zst"

# Quantization constants -- MUST match the C++ side.
QA, QB, SCALE = 255, 64, 400

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

In [ ]:
PIECE_TO_TYPE = {p: i for i, p in enumerate("pnbrqk")}

def fen_to_features(fen: str):
    """Return (white_persp_indices, black_persp_indices, stm).
    stm: 0 = white to move, 1 = black to move."""
    parts = fen.split(" ")
    board_part, stm_part = parts[0], parts[1]

    w_idx, b_idx = [], []
    sq = 56                                # FEN starts at a8; a8 = 56 in 0..63
    for ch in board_part:
        if ch == "/":
            sq -= 16                       # drop one rank, back to file a
        elif ch.isdigit():
            sq += int(ch)                  # skip empty squares
        else:
            color = 0 if ch.isupper() else 1        # 0 = white piece
            pt = PIECE_TO_TYPE[ch.lower()]
            # White perspective: white pieces are "own" (is_enemy = color)
            w_idx.append(color * 384 + pt * 64 + sq)
            # Black perspective: black pieces are "own", board flipped
            b_idx.append((1 - color) * 384 + pt * 64 + (sq ^ 56))
            sq += 1

    return w_idx, b_idx, 0 if stm_part == "w" else 1

# quick self-test on the start position
_w, _b, _s = fen_to_features(
    "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1")
assert len(_w) == 32 and len(_b) == 32 and _s == 0
assert sorted(_w) == sorted(_b)   # start position is symmetric
print("FEN parser OK")


FEN parser OK


In [ ]:
import json, time
import numpy as np
import requests, zstandard

MAX_FEATURES = 32   # at most 32 pieces on the board; pad shorter lists
PAD_INDEX    = 768  # dummy feature index used for padding (row is all-zero)

def stream_positions(url: str, limit: int):
    """Yield (w_idx, b_idx, stm, cp_stm) tuples, decompressing over HTTP."""
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        dctx = zstandard.ZstdDecompressor()
        stream = dctx.stream_reader(r.raw)
        buf = b""
        produced = 0
        while produced < limit:
            chunk = stream.read(1 << 20)          # 1 MB at a time
            if not chunk:
                break
            buf += chunk
            *lines, buf = buf.split(b"\n")        # keep trailing partial line
            for line in lines:
                if produced >= limit:
                    break
                try:
                    obj = json.loads(line)
                    best = max(obj["evals"], key=lambda e: e["depth"])
                    pv = best["pvs"][0]
                    if "cp" in pv:
                        cp = max(-CP_CLAMP, min(CP_CLAMP, pv["cp"]))
                    else:                          # mate score
                        cp = CP_CLAMP if pv["mate"] > 0 else -CP_CLAMP
                    w, b, stm = fen_to_features(obj["fen"])
                    # The Lichess DB contains a few corrupt/illegal FENs
                    # (e.g. >32 pieces).  Legal chess never exceeds 32, so
                    # anything bigger is garbage -- skip it.
                    if len(w) > MAX_FEATURES:
                        continue
                    if stm == 1:                   # side-to-move relative
                        cp = -cp
                    yield w, b, stm, cp
                    produced += 1
                except (KeyError, ValueError, IndexError):
                    continue                       # skip malformed lines

def preprocess():
    Wf = np.full((SHARD_SIZE, MAX_FEATURES), PAD_INDEX, dtype=np.int16)
    Bf = np.full((SHARD_SIZE, MAX_FEATURES), PAD_INDEX, dtype=np.int16)
    St = np.zeros(SHARD_SIZE, dtype=np.uint8)
    Cp = np.zeros(SHARD_SIZE, dtype=np.int16)

    n_in_shard, shard_id, total = 0, 0, 0
    t0 = time.time()

    for w, b, stm, cp in stream_positions(LICHESS_URL, N_POSITIONS):
        Wf[n_in_shard, :len(w)] = w
        Bf[n_in_shard, :len(b)] = b
        St[n_in_shard] = stm
        Cp[n_in_shard] = cp
        n_in_shard += 1
        total += 1

        if n_in_shard == SHARD_SIZE:
            np.savez_compressed(f"{DATA_DIR}/shard_{shard_id:03d}.npz",
                                w=Wf, b=Bf, stm=St, cp=Cp)
            print(f"shard {shard_id} saved | {total:,} positions | "
                  f"{total / (time.time() - t0):,.0f} pos/s")
            shard_id += 1
            n_in_shard = 0
            Wf.fill(PAD_INDEX); Bf.fill(PAD_INDEX)

    if n_in_shard:                                 # final partial shard
        np.savez_compressed(f"{DATA_DIR}/shard_{shard_id:03d}.npz",
                            w=Wf[:n_in_shard], b=Bf[:n_in_shard],
                            stm=St[:n_in_shard], cp=Cp[:n_in_shard])
    print(f"done: {total:,} positions in {time.time() - t0:,.0f}s")

# Skip if shards already exist (lets you re-run the notebook after a restart)
if not os.listdir(DATA_DIR):
    preprocess()
else:
    print("shards already present, skipping preprocessing")

shards already present, skipping preprocessing


In [ ]:
import torch

SHARD_FILES = sorted(os.path.join(DATA_DIR, f) for f in os.listdir(DATA_DIR))
assert len(SHARD_FILES) >= 2, "need at least 2 shards -- run Cell 4 first"

VAL_SHARD    = SHARD_FILES[-1]     # held out entirely for validation
TRAIN_SHARDS = SHARD_FILES[:-1]

def load_shard(path):
    """Load one shard as CPU tensors: features, side-to-move, sigmoid target."""
    d = np.load(path)
    return (torch.from_numpy(d["w"]),
            torch.from_numpy(d["b"]),
            torch.from_numpy(d["stm"]),
            torch.sigmoid(torch.from_numpy(d["cp"]).float() / SCALE))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"{len(TRAIN_SHARDS)} train shards + 1 val shard | device: {device}")


9 train shards + 1 val shard | device: cuda


In [ ]:
import torch.nn as nn

class NNUE(nn.Module):
    def __init__(self, hidden: int = HIDDEN):
        super().__init__()
        # 769 rows: 768 real features + 1 zero padding row
        self.ft = nn.EmbeddingBag(769, hidden, mode="sum", padding_idx=PAD_INDEX)
        self.ft_bias = nn.Parameter(torch.zeros(hidden))
        self.out = nn.Linear(2 * hidden, 1)
        # small init keeps early sigmoid outputs near 0.5
        nn.init.uniform_(self.ft.weight, -0.05, 0.05)
        with torch.no_grad():
            self.ft.weight[PAD_INDEX].zero_()

    def forward(self, w_idx, b_idx, stm):
        # accumulators for both perspectives (shared weights)
        acc_w = self.ft(w_idx) + self.ft_bias
        acc_b = self.ft(b_idx) + self.ft_bias
        # order as [side-to-move, opponent]
        stm = stm.unsqueeze(1).bool()               # True where black to move
        us   = torch.where(stm, acc_b, acc_w)
        them = torch.where(stm, acc_w, acc_b)
        x = torch.cat([us, them], dim=1).clamp(0, 1)   # Clipped ReLU
        return self.out(x).squeeze(1)                  # ~ cp / SCALE

model = NNUE().to(device)
print(sum(p.numel() for p in model.parameters()), "parameters")


197633 parameters


In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=LR)
sched = torch.optim.lr_scheduler.StepLR(opt, step_size=LR_DROP_EVERY, gamma=1/3)
loss_fn = nn.MSELoss()

def run_batch(W, B, S, T, idx, train: bool):
    w   = W[idx].to(device).long()   # EmbeddingBag wants int64
    b   = B[idx].to(device).long()
    stm = S[idx].to(device)
    tgt = T[idx].to(device)
    pred = torch.sigmoid(model(w, b, stm))
    loss = loss_fn(pred, tgt)
    if train:
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        # padding row must stay zero (padding_idx blocks grads, but be safe)
        with torch.no_grad():
            model.ft.weight[PAD_INDEX].zero_()
    return loss.item()

torch.manual_seed(SEED)
np.random.seed(SEED)

# Load the validation shard ONCE (it never changes); cap for quick eval.
Wv, Bv, Sv, Tv = load_shard(VAL_SHARD)
n_val = min(len(Sv), VAL_MAX)
print(f"validating on {n_val:,} held-out positions")

for epoch in range(1, EPOCHS + 1):
    model.train()
    tot, nb = 0.0, 0
    for si in np.random.permutation(len(TRAIN_SHARDS)):      # shard order
        W, B, S, T = load_shard(TRAIN_SHARDS[si])             # ~150 MB
        perm = torch.randperm(len(S))                         # within-shard
        for i in range(0, len(S), BATCH_SIZE):
            tot += run_batch(W, B, S, T, perm[i:i + BATCH_SIZE], train=True)
            nb += 1
    sched.step()

    model.eval()
    with torch.no_grad():
        vtot, vnb = 0.0, 0
        for i in range(0, n_val, BATCH_SIZE):
            vtot += run_batch(Wv, Bv, Sv, Tv,
                              torch.arange(i, min(i + BATCH_SIZE, n_val)),
                              train=False)
            vnb += 1

    print(f"epoch {epoch:3d} | train {tot/nb:.5f} | val {vtot/max(vnb,1):.5f} "
          f"| lr {sched.get_last_lr()[0]:.1e}")
    torch.save({"model": model.state_dict(), "epoch": epoch},
               f"{CKPT_DIR}/ckpt_latest.pt")


validating on 200,000 held-out positions
epoch   1 | train 0.02494 | val 0.02130 | lr 1.0e-03
epoch   2 | train 0.01996 | val 0.02007 | lr 1.0e-03
epoch   3 | train 0.01869 | val 0.01983 | lr 1.0e-03
epoch   4 | train 0.01797 | val 0.01960 | lr 1.0e-03
epoch   5 | train 0.01748 | val 0.01952 | lr 1.0e-03
epoch   6 | train 0.01712 | val 0.01939 | lr 1.0e-03
epoch   7 | train 0.01685 | val 0.01933 | lr 1.0e-03
epoch   8 | train 0.01657 | val 0.01940 | lr 3.3e-04
epoch   9 | train 0.01619 | val 0.01914 | lr 3.3e-04
epoch  10 | train 0.01601 | val 0.01906 | lr 3.3e-04
epoch  11 | train 0.01592 | val 0.01898 | lr 3.3e-04
epoch  12 | train 0.01581 | val 0.01903 | lr 3.3e-04
epoch  13 | train 0.01572 | val 0.01910 | lr 3.3e-04
epoch  14 | train 0.01568 | val 0.01911 | lr 3.3e-04
epoch  15 | train 0.01560 | val 0.01906 | lr 3.3e-04
epoch  16 | train 0.01556 | val 0.01907 | lr 1.1e-04
epoch  17 | train 0.01542 | val 0.01900 | lr 1.1e-04
epoch  18 | train 0.01537 | val 0.01896 | lr 1.1e-04
epoch

In [ ]:
import struct

def export(model: NNUE, path: str):
    ft_w  = model.ft.weight.detach().cpu().numpy()[:768]     # drop pad row
    ft_b  = model.ft_bias.detach().cpu().numpy()
    out_w = model.out.weight.detach().cpu().numpy().reshape(-1)
    out_b = float(model.out.bias.detach().cpu().numpy()[0])

    ft_w_q  = np.round(ft_w  * QA).astype(np.int64)
    ft_b_q  = np.round(ft_b  * QA).astype(np.int64)
    out_w_q = np.round(out_w * QB).astype(np.int64)
    out_b_q = int(round(out_b * QA * QB))

    # verify everything fits its integer type before truncating
    for name, arr in [("ft_w", ft_w_q), ("ft_b", ft_b_q), ("out_w", out_w_q)]:
        assert np.abs(arr).max() < 32768, f"{name} overflows int16!"

    with open(path, "wb") as f:
        f.write(b"MNNU")
        f.write(struct.pack("<II", 1, HIDDEN))                 # version, hidden
        f.write(ft_w_q.astype("<i2").tobytes())                # [768][HIDDEN]
        f.write(ft_b_q.astype("<i2").tobytes())                # [HIDDEN]
        f.write(out_w_q.astype("<i2").tobytes())               # [2*HIDDEN]
        f.write(struct.pack("<i", out_b_q))                    # i32 bias
    print(f"wrote {path}  ({os.path.getsize(path)/1024:.0f} KB)")

export(model, OUT_FILE)

wrote /content/mihir_v1.nnue  (386 KB)


In [ ]:
def quantized_eval(fen: str, model: NNUE) -> int:
    ft_w  = np.round(model.ft.weight.detach().cpu().numpy()[:768] * QA)
    ft_b  = np.round(model.ft_bias.detach().cpu().numpy() * QA)
    out_w = np.round(model.out.weight.detach().cpu().numpy().reshape(-1) * QB)
    out_b = round(float(model.out.bias.detach().cpu()) * QA * QB)

    w, b, stm = fen_to_features(fen)
    acc_w = ft_b + ft_w[w].sum(axis=0)
    acc_b = ft_b + ft_w[b].sum(axis=0)
    us, them = (acc_b, acc_w) if stm else (acc_w, acc_b)
    act = np.concatenate([np.clip(us, 0, QA), np.clip(them, 0, QA)])
    raw = out_b + int((act * out_w).sum())
    return raw * SCALE // (QA * QB)          # centipawns, stm-relative

def float_eval(fen: str, model: NNUE) -> float:
    w, b, stm = fen_to_features(fen)
    w += [PAD_INDEX] * (MAX_FEATURES - len(w))
    b += [PAD_INDEX] * (MAX_FEATURES - len(b))
    with torch.no_grad():
        v = model(torch.tensor([w], device=device),
                  torch.tensor([b], device=device),
                  torch.tensor([stm], device=device))
    return float(v) * SCALE

model.eval()
for fen in [
    "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1",
    "rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq - 0 1",
    "8/8/8/8/3k4/8/3P4/3K4 w - - 0 1",
    "r1bqkbnr/pppp1ppp/2n5/4p3/2B1P3/5N2/PPPP1PPP/RNBQK2R b KQkq - 3 3",
]:
    print(f"{fen[:40]:42s} float {float_eval(fen, model):+8.1f} cp | "
          f"quant {quantized_eval(fen, model):+5d} cp")


rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQK   float    +30.4 cp | quant   +30 cp
rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNB   float    -30.4 cp | quant   -30 cp
8/8/8/8/3k4/8/3P4/3K4 w - - 0 1            float   +465.1 cp | quant  +478 cp
r1bqkbnr/pppp1ppp/2n5/4p3/2B1P3/5N2/PPPP   float     +1.8 cp | quant    +0 cp


In [ ]:
from google.colab import files
files.download(OUT_FILE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>